# Task 4 — Voiceprint Verification Model

**Goal:** given a voice sample, decide whether it belongs to the person the face model just recognised.

**Approach**
1. Train a speaker classifier on `audio_features.csv` (MFCCs, roll-off, spectral and energy statistics).
2. Evaluate with **leave-one-recording-out** cross-validation, grouped by `source_file`, so an augmented copy of a
   recording is never in train while its original is in test.
3. Report **accuracy, macro F1 and log-loss** (the metrics the brief asks for).
4. Build a **voiceprint** per member (mean standardised feature vector) and a **distance threshold** so unknown voices are
   rejected, then wrap everything in `verify_voice()` for the command-line app.

`duration_s` and `energy` are excluded from training: they depend on the phrase and recording volume, not on who is speaking.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import classification_report

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts import voice_model as vm

## 2. Load the audio feature table

In [ ]:
df, feature_cols = vm.load_data(ROOT / "audio_features.csv")
print(f"{len(df)} rows | {df.member.nunique()} members | {df.source_file.nunique()} recordings | {len(feature_cols)} features used")
display(pd.crosstab(df.member, df.augmentation))

## 3. Compare models (leave-one-recording-out)

Each fold hides **both the original and all augmented versions of one recording** and trains on the other 7.
The held-out recording is also a *different phrase* from the member's other recording, so this is a deliberately
demanding test. Two models are compared; the best by macro-F1 (ties broken by lower log-loss) is selected.

In [ ]:
table, results = vm.compare_models(df, feature_cols)
display(table.round(3))
best = results[table.iloc[0].model]
print("Selected:", best["model"])
print("Originals only:", {k: round(v, 3) for k, v in best["originals_only"].items()})

### Confusion matrix and per-member report

In [ ]:
fig = vm.plot_confusion(df, best, ROOT / "reports" / "voice_confusion_matrix.png")
plt.show()
pred = best["classes"][best["proba"].argmax(axis=1)]
print(classification_report(df.member, pred, zero_division=0))

### Reading the results  *(complete after running)*
* Which member(s), if any, are confused with each other, and are their pitch/centroid values close (see the Task 3 summary table)?
* Is accuracy on **originals only** noticeably different from all rows? A large gap would mean augmentation changes the voice too much.
* With only 2 recordings per member each fold has a single held-out recording, so these metrics are **noisy** — state this in the report.

## 4. Train the final model, set the distance threshold, save

`data/unauthorized/` should contain recordings of people **not** in the team. If it exists, the threshold is calibrated on
genuine (held-out) vs. unauthorized distances to the voiceprint; otherwise a provisional value is used and clearly flagged.
A classifier's confidence is *not* used to spot strangers because it always favours its closest member.

In [ ]:
bundle = vm.train_and_save(
    csv_path=ROOT / "audio_features.csv",
    model_path=ROOT / "models" / "voice_model.joblib",
    unauthorized_dir=ROOT / "data" / "unauthorized",
    report_dir=ROOT / "reports",
)

## 5. Verification demo

In [ ]:
bundle = vm.load_bundle(ROOT / "models" / "voice_model.joblib")
audio = vm.af.list_audio_files(ROOT / "data" / "audio")

def show(r): print(f"{'APPROVED' if r['approved'] else 'DENIED  '} | claimed={r['claimed_member']:<8} predicted={r['predicted_member']:<8} conf={r['confidence']:.2f} dist={r['distance_to_claimed']}/{r['distance_threshold']} | {r['reason']}")

# 1) genuine: each member's own recording, claiming their own identity
for f in audio[:4]:
    member, _ = vm.af.parse_filename(f)
    show(vm.verify_voice(f, member, bundle))

In [ ]:
# 2) wrong claim: someone's voice while claiming to be somebody else
a, b = audio[0], audio[-1]
claimed = vm.af.parse_filename(b)[0]
print(f"{a.name} claiming to be {claimed}:")
show(vm.verify_voice(a, claimed, bundle))

In [ ]:
# 3) unauthorized voice (not enrolled) claiming to be a member
unauth = vm.af.list_audio_files(ROOT / "data" / "unauthorized") if (ROOT / "data" / "unauthorized").exists() else []
if unauth:
    for f in unauth:
        show(vm.verify_voice(f, vm.af.parse_filename(audio[0])[0], bundle))
else:
    print("No recordings in data/unauthorized/ yet — add one or two (Task 6 needs this demo).")

## 6. Limitations to state in the report
* Only 2 recordings per member: metrics are indicative, not a precise estimate of real-world accuracy.
* The model recognises **who** is speaking, not **what** is said, so it does not check the phrase itself (that would need speech recognition).
* A closed-set classifier always picks its closest member, so rejection relies on the voiceprint-distance threshold. Calibrate it with real unauthorized recordings; the provisional value is only a starting point.
* Recordings from other microphones/rooms will shift the features; augmentation (noise, pitch, speed) only partly covers that.